# Base YOLOv8 (Nano)

## Import Libraries

In [1]:
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision import models
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import torch.optim as optim
# Annotation Conversion
import xml.etree.ElementTree as ET
import os
import shutil
import pandas as pd
from ultralytics import YOLO
# Calculate FLOPs
from thop import profile

# torch GPU Usage
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f'Using {device}')

Using cuda


## Model Training

In [2]:
def main():
    torch.cuda.empty_cache() # Clear memory
    model = YOLO("yolov8n.pt") # Load model
    # Training
    train_results = model.train(
        data="config.yaml",
        epochs=20,
        imgsz=640,
        batch=24,
        workers=5,
        device=0,
        name="train"
    )

if __name__ == "__main__":
    main()

New https://pypi.org/project/ultralytics/8.4.35 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.33  Python-3.10.20 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=24, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=config.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train4, nbs=

In [3]:
# Load the trained model
model_path = "runs/detect/train4/weights/best.pt"
model = YOLO(model_path)

# Get model info
size_mb = os.path.getsize(model_path) / (1024 * 1024)
params = sum(p.numel() for p in model.model.parameters())
flops = model.flops / 1e9 if hasattr(model, "flops") else None

# Epochs of last metric (val)
df = pd.read_csv("runs/detect/train4/results.csv")
last = df.iloc[-1]

# Calc F1 score
precisionGet = last.get("metrics/precision(B)")
recallGet = last.get("metrics/recall(B)")

# Calc GFLOPs
model.model.to(device)
model.model.eval()
dummy = torch.randn(1, 3, 640, 640, device=device) # Dummy input
# Model profiling
macs, params = profile(model.model, inputs=(dummy,), verbose=False) # Add custom ops when custom
gflops = macs / 1e9

# Summary of metrics
summary = {
    "Size": f"{size_mb:.2f} MB",
    "Params": params,
    "mAP_50": last.get("metrics/mAP50(B)"),
    "mAP_50-95": last.get("metrics/mAP50-95(B)"),
    "F1": round(2 * (precisionGet * recallGet) / (precisionGet + recallGet) if (precisionGet + recallGet) > 0 else 0, 4),
    "precision": precisionGet,
    "recall": recallGet,
    "GFLOPs": gflops,
    "train_loss": last.get("train/box_loss") + last.get("train/cls_loss"),
    "val_loss": last.get("val/box_loss") + last.get("val/cls_loss")
}

print("\nMODEL SUMMARY")
for k, v in summary.items():
    if isinstance(v, float):
        v = round(v, 4)
    print(f"{k:15}: {v}")


MODEL SUMMARY
Size           : 5.96 MB
Params         : 3011823.0
mAP_50         : 0.4971
mAP_50-95      : 0.2583
F1             : 0.5087
precision      : 0.5408
recall         : 0.4802
GFLOPs         : 4.0992
train_loss     : 3.4656
val_loss       : 3.6589
